# Direct classification

In this notebook we attempt the following improvements to obtain a better accuracy score.

- Directly classifying health status from gut biome, rather than a two-step pipeline
- Accounting for the compositional nature of the microbiome data using a central log-ratio (CLR) transformation
- Over/under-sampling or weighting to account for class imbalance during training.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import lightgbm as lgb
from tqdm.auto import tqdm, trange

pd.set_option("display.precision", 3)
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Data loading

Load the pre-processed AGP dataset: `meta.tsv` (id, age, health) and `otu.tsv` (~5,966 samples × 942 OTUs after 10% prevalence filtering — see `prepare_data.py`).

The paper's GAI pipeline trains a regression model on the **healthy** cohort, then derives a Gut Age Index for classification. Here we skip that indirection and classify **directly** from OTU abundances.

In [ ]:
DATA = "data/processed/AGP"
meta = pd.read_csv(f"{DATA}/meta.tsv", sep="\t", index_col="id")
otu  = pd.read_csv(f"{DATA}/otu.tsv",  sep="\t", index_col="id")

# Align indices
common = meta.index.intersection(otu.index)
meta, otu = meta.loc[common], otu.loc[common]

y = (meta["health"] == "n").astype(int).values
print(f"Samples: {len(y)}  |  Healthy: {(y==0).sum()}  |  Non-healthy: {(y==1).sum()}  |  Prevalence: {y.mean():.1%}")
print(f"OTU features: {otu.shape[1]}")

## 2. CLR transformation

Microbiome counts are **compositional** — relative abundances sum to a constant per sample, so standard Euclidean distances are misleading (Aitchison, 1986). The Centered Log-Ratio addresses this:

$$\text{CLR}(x_i) = \ln(x_i + 1) - \frac{1}{D}\sum_{j=1}^{D} \ln(x_j + 1)$$

The pseudocount of 1.0 handles the many zeros in sparse OTU data. After CLR, each sample's transformed values sum to zero — the data lives in a proper Euclidean subspace where standard ML methods apply.

In [ ]:
def clr_transform(df, pseudocount=1.0):
    """Centered log-ratio transform. Rows = samples, cols = OTUs."""
    log_mat = np.log(df.values + pseudocount)
    return log_mat - log_mat.mean(axis=1, keepdims=True)

X = clr_transform(otu)
print(f"X shape: {X.shape}")
print(f"Row sums ≈ 0 check: max |row sum| = {np.abs(X.sum(axis=1)).max():.2e}")

## 3. Model definitions

Three classifiers of increasing complexity, all handling the ~2:1 class imbalance via balanced weights:

| Model | Type | Rationale |
|---|---|---|
| **LightGBM** | Gradient boosting | Fast tree ensemble — handles high-dim sparse features natively |
| **Random Forest** | Ensemble of trees | Nonlinear, robust to correlated & high-dimensional features |
| **MLP** | Neural network | Can learn complex OTU interaction patterns |

In [ ]:
class MicrobiomeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BaselineMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2),
        )
    def forward(self, x):
        return self.net(x)

## 4. Training helpers

The MLP uses a manual training loop with weighted `CrossEntropyLoss` to handle the ~2:1 class imbalance. Scikit-learn models use `class_weight='balanced'` for the same purpose.

In [ ]:
def train_mlp(X_tr, y_tr, X_val, y_val, n_epochs=10, lr=1e-3, batch_size=256):
    """Train MLP for one fold, return val_probs."""
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr)
    weights = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

    model = BaselineMLP(X_tr.shape[1]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    train_ds = MicrobiomeDataset(X_tr, y_tr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    for epoch in trange(n_epochs, desc="    MLP epochs", leave=False):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        logits = model(xv)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    return probs

## 5. Feature ablation: CLR × age

We evaluate all four combinations of {Raw OTU counts, CLR-transformed} × {microbiome only, microbiome + age} across the three models. This tells us (a) whether CLR helps over raw counts, and (b) whether age adds discriminative power.

Age is standardised per fold (fit on train only) to avoid leakage. It bypasses CLR since it is not compositional.

In [ ]:
from sklearn.preprocessing import StandardScaler

model_names = ["LightGBM", "Random Forest", "MLP"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

age_raw = meta["age"].values.reshape(-1, 1)
X_clr = clr_transform(otu)
X_raw = otu.values

feature_set_names = ["Raw", "Raw + age", "CLR", "CLR + age"]

# Store results: feature_set → model → {ba, auc, oof_probs}
ablation = {
    fs: {name: {"ba": [], "auc": [], "oof_probs": np.zeros(len(y))} for name in model_names}
    for fs in feature_set_names
}

for fold, (tr_idx, val_idx) in enumerate(tqdm(list(cv.split(X_clr, y)), desc="Ablation folds")):
    age_scaler = StandardScaler().fit(age_raw[tr_idx])
    age_tr = age_scaler.transform(age_raw[tr_idx])
    age_val = age_scaler.transform(age_raw[val_idx])

    fold_X = {
        "Raw":       (X_raw[tr_idx], X_raw[val_idx]),
        "Raw + age": (np.column_stack([X_raw[tr_idx], age_tr]),
                      np.column_stack([X_raw[val_idx], age_val])),
        "CLR":       (X_clr[tr_idx], X_clr[val_idx]),
        "CLR + age": (np.column_stack([X_clr[tr_idx], age_tr]),
                      np.column_stack([X_clr[val_idx], age_val])),
    }
    y_tr, y_val = y[tr_idx], y[val_idx]

    for fs_name in tqdm(feature_set_names, desc=f"  Fold {fold}", leave=False):
        Xf_tr, Xf_val = fold_X[fs_name]
        for name in model_names:
            if name == "MLP":
                probs = train_mlp(Xf_tr, y_tr, Xf_val, y_val)
            elif name == "Random Forest":
                clf = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED)
                clf.fit(Xf_tr, y_tr)
                probs = clf.predict_proba(Xf_val)[:, 1]
            elif name == "LightGBM":
                clf = lgb.LGBMClassifier(is_unbalance=True, n_jobs=-1, random_state=SEED, verbosity=-1)
                clf.fit(Xf_tr, y_tr)
                probs = clf.predict_proba(Xf_val)[:, 1]

            preds = (probs >= 0.5).astype(int)
            ablation[fs_name][name]["ba"].append(balanced_accuracy_score(y_val, preds))
            ablation[fs_name][name]["auc"].append(roc_auc_score(y_val, probs))
            ablation[fs_name][name]["oof_probs"][val_idx] = probs

## 6. Ablation results

Paper baseline: **68% balanced accuracy** (GAI thresholding on the same AGP cohort). Our target: **BA > 70%, AUC > 0.75**.

In [ ]:
# Consolidated results table (mean ± std over 5 folds)
rows = []
for fs_name in feature_set_names:
    for name in model_names:
        ba = np.array(ablation[fs_name][name]["ba"])
        auc_arr = np.array(ablation[fs_name][name]["auc"])
        rows.append({
            "Features": fs_name,
            "Model": name,
            "Balanced Accuracy": f"{ba.mean():.3f} ± {ba.std():.3f}",
            "AUC-ROC": f"{auc_arr.mean():.3f} ± {auc_arr.std():.3f}",
        })
pd.DataFrame(rows)

In [ ]:
# 2×2 ROC plot: rows = {Raw, CLR}, cols = {no age, + age}
fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)
colors = {"LightGBM": "#4e79a7", "Random Forest": "#59a14f", "MLP": "#e15759"}

layout = [
    ("Raw",       axes[0, 0]), ("Raw + age", axes[0, 1]),
    ("CLR",       axes[1, 0]), ("CLR + age", axes[1, 1]),
]

for fs_name, ax in layout:
    for name in model_names:
        oof = ablation[fs_name][name]["oof_probs"]
        fpr, tpr, _ = roc_curve(y, oof)
        auc_val = roc_auc_score(y, oof)
        ax.plot(fpr, tpr, color=colors[name], label=f"{name} ({auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.5)
    ax.set_title(fs_name, fontsize=11)
    ax.legend(fontsize=7, frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)

axes[1, 0].set_xlabel("FPR"); axes[1, 1].set_xlabel("FPR")
axes[0, 0].set_ylabel("TPR"); axes[1, 0].set_ylabel("TPR")
fig.suptitle("ROC — feature ablation (default hyperparameters)", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Tufte-style dot-and-whisker: AUC per feature set × model
fig, ax = plt.subplots(figsize=(7, 3.5))
colors = {"LightGBM": "#4e79a7", "Random Forest": "#59a14f", "MLP": "#e15759"}

y_pos = np.arange(len(feature_set_names))
offsets = {"LightGBM": -0.18, "Random Forest": 0.0, "MLP": 0.18}

for name in model_names:
    means, stds = [], []
    fold_vals = []
    for fs_name in feature_set_names:
        vals = ablation[fs_name][name]["auc"]
        means.append(np.mean(vals))
        stds.append(np.std(vals))
        fold_vals.append(vals)

    yp = y_pos + offsets[name]
    # Thin whiskers: mean ± std
    ax.errorbar(means, yp, xerr=stds, fmt="none", ecolor=colors[name],
                capsize=0, elinewidth=1.2, alpha=0.8)
    # Individual fold values (small dots)
    for i, fv in enumerate(fold_vals):
        ax.scatter(fv, [yp[i]] * len(fv), color=colors[name], s=14, alpha=0.45, zorder=3)
    # Mean (larger dot)
    ax.scatter(means, yp, color=colors[name], s=55, zorder=4, label=name,
               edgecolors="white", linewidth=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(feature_set_names)
ax.set_xlabel("AUC-ROC")
ax.set_title("AUC by feature set and model (5-fold CV, mean ± std)")
ax.legend(frameon=False, loc="lower right", fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.axvline(0.5, color="grey", ls=":", lw=0.5)
fig.tight_layout()
plt.show()

In [ ]:
# Choose feature set for downstream hyperparameter tuning
USE_CLR = True   # Set to False to use raw OTU counts
USE_AGE = True   # Set to False to drop age

best_fs = ("CLR" if USE_CLR else "Raw") + (" + age" if USE_AGE else "")
print(f"Selected feature set for tuning: {best_fs}")

if USE_CLR:
    X = clr_transform(otu)
else:
    X = otu.values

if USE_AGE:
    _age_scaler = StandardScaler().fit(age_raw)
    X = np.column_stack([X, _age_scaler.transform(age_raw)])

print(f"X shape: {X.shape}")

## 7. Hyperparameter tuning (sklearn-compatible models)

We tune Random Forest and LightGBM via `RandomizedSearchCV` with 2-fold inner CV (20 random configs each). The MLP keeps its fixed architecture — its ablation results carry forward unchanged.

In [ ]:
rf_param_dist = {
    "n_estimators": randint(200, 1000),
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", 0.1, 0.2, 0.3],
}

lgb_param_dist = {
    "n_estimators": randint(200, 1000),
    "max_depth": [3, 5, 7, 10, 15, -1],
    "learning_rate": uniform(0.01, 0.29),
    "num_leaves": randint(15, 127),
    "subsample": uniform(0.5, 0.5),
    "colsample_bytree": uniform(0.3, 0.7),
}

sklearn_models = ["LightGBM", "Random Forest"]

def make_tuned(estimator, param_dist, X_tr, y_tr):
    """Tune via inner CV on training data only, return fitted best estimator."""
    search = RandomizedSearchCV(
        estimator, param_dist, n_iter=20, cv=2, scoring="balanced_accuracy",
        n_jobs=-1, random_state=SEED, error_score="raise",
    )
    search.fit(X_tr, y_tr)
    return search.best_estimator_

In [ ]:
# Carry MLP results forward from ablation (no tuning for MLP)
tuned_results = {
    "MLP": ablation[best_fs]["MLP"],
}
best_params = {"MLP": []}

for name in sklearn_models:
    tuned_results[name] = {"ba": [], "auc": [], "oof_probs": np.zeros(len(y))}
    best_params[name] = []

for fold, (tr_idx, val_idx) in enumerate(tqdm(list(cv.split(X, y)), desc="Tuning folds")):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    for name in sklearn_models:
        if name == "Random Forest":
            clf = make_tuned(
                RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
                rf_param_dist, X_tr, y_tr,
            )
        elif name == "LightGBM":
            clf = make_tuned(
                lgb.LGBMClassifier(is_unbalance=True, n_jobs=-1, random_state=SEED, verbosity=-1),
                lgb_param_dist, X_tr, y_tr,
            )
        probs = clf.predict_proba(X_val)[:, 1]
        best_params[name].append(clf.get_params())
        preds = (probs >= 0.5).astype(int)
        tuned_results[name]["ba"].append(balanced_accuracy_score(y_val, preds))
        tuned_results[name]["auc"].append(roc_auc_score(y_val, probs))
        tuned_results[name]["oof_probs"][val_idx] = probs

    print(f"Fold {fold}: " + " | ".join(
        f"{n}: BA={tuned_results[n]['ba'][-1]:.3f} AUC={tuned_results[n]['auc'][-1]:.3f}" for n in sklearn_models
    ))

## 8. Tuned results — default vs tuned comparison

In [ ]:
# Default vs Tuned comparison table
comp_rows = []
for name in model_names:
    b_ba  = np.mean(ablation[best_fs][name]["ba"])
    b_auc = np.mean(ablation[best_fs][name]["auc"])
    t_ba  = np.mean(tuned_results[name]["ba"])
    t_auc = np.mean(tuned_results[name]["auc"])
    b_ba_std  = np.std(ablation[best_fs][name]["ba"])
    t_ba_std  = np.std(tuned_results[name]["ba"])
    b_auc_std = np.std(ablation[best_fs][name]["auc"])
    t_auc_std = np.std(tuned_results[name]["auc"])
    comp_rows.append({
        "Model": name,
        "Default BA": f"{b_ba:.3f} ± {b_ba_std:.3f}",
        "Tuned BA":   f"{t_ba:.3f} ± {t_ba_std:.3f}",
        "Δ BA":       f"{t_ba - b_ba:+.3f}",
        "Default AUC": f"{b_auc:.3f} ± {b_auc_std:.3f}",
        "Tuned AUC":   f"{t_auc:.3f} ± {t_auc_std:.3f}",
        "Δ AUC":       f"{t_auc - b_auc:+.3f}",
    })
pd.DataFrame(comp_rows)

In [ ]:
# ROC: default vs tuned (side by side)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
colors = {"LightGBM": "#4e79a7", "Random Forest": "#59a14f", "MLP": "#e15759"}

for ax, (label, src) in zip(axes, [("Default", {n: ablation[best_fs][n] for n in model_names}),
                                     ("Tuned", tuned_results)]):
    for name in model_names:
        oof = src[name]["oof_probs"]
        fpr, tpr, _ = roc_curve(y, oof)
        auc_val = roc_auc_score(y, oof)
        ax.plot(fpr, tpr, color=colors[name], label=f"{name} ({auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.5)
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=7, frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")

fig.suptitle(f"ROC — {best_fs} features", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

# Selected hyperparameters per fold
for name in sklearn_models:
    if not best_params.get(name):
        continue
    keys = [k for k in best_params[name][0] if k not in ("random_state", "n_jobs", "verbose", "class_weight")]
    param_df = pd.DataFrame([{k: p[k] for k in keys} for p in best_params[name]])
    print(f"\n{name} — tuned params across folds:")
    display(param_df)